In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("marketing_attribution_dataset.csv")

# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Sort properly (VERY IMPORTANT)
df = df.sort_values(by=['user_id', 'timestamp'])

print(df.head())

   user_id  touchpoint_order     channel           timestamp  conversion
3        1                 4    Referral 2026-05-02 05:00:00           0
4        1                 5      Direct 2026-05-07 05:00:00           1
0        1                 1  Google Ads 2026-05-09 06:00:00           0
2        1                 3      Direct 2026-05-11 01:00:00           0
1        1                 2  Google Ads 2026-05-11 16:00:00           0


In [ ]:
# Group journeys
journeys = df.groupby('user_id').apply(
    lambda x: list(x.sort_values('timestamp')['channel'])
).reset_index(name='journey')

# Identify converters
conversions = df.groupby('user_id')['conversion'].max().reset_index()

# Merge
journeys = journeys.merge(conversions, on='user_id')

print(journeys.head())

   user_id                                            journey  conversion
0        1  [Referral, Direct, Google Ads, Direct, Google ...           1
1        2                 [Direct, Organic Search, Referral]           1
2        3  [Email, Referral, Google Ads, Referral, Social...           1
3        4  [Google Ads, Google Ads, Organic Search, Refer...           1
4        5                            [Organic Search, Email]           1


/tmp/ipykernel_2081/3985091157.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  journeys = df.groupby('user_id').apply(


In [ ]:
def first_touch_attribution(journeys):
    results = {}

    for _, row in journeys.iterrows():
        if row['conversion'] == 1:
            first_channel = row['journey'][0]
            results[first_channel] = results.get(first_channel, 0) + 1

    return pd.DataFrame(list(results.items()), columns=['channel', 'credit'])

In [ ]:
def last_touch_attribution(journeys):
    results = {}

    for _, row in journeys.iterrows():
        if row['conversion'] == 1:
            last_channel = row['journey'][-1]
            results[last_channel] = results.get(last_channel, 0) + 1

    return pd.DataFrame(list(results.items()), columns=['channel', 'credit'])

In [ ]:
def linear_attribution(journeys):
    results = {}

    for _, row in journeys.iterrows():
        if row['conversion'] == 1:
            path = row['journey']
            credit = 1 / len(path)

            for channel in path:
                results[channel] = results.get(channel, 0) + credit

    return pd.DataFrame(list(results.items()), columns=['channel', 'credit'])

In [ ]:
def time_decay_attribution(df):
    results = {}

    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('timestamp')

        if group['conversion'].max() == 1:
            n = len(group)
            weights = np.exp(np.linspace(-1, 0, n))  # decay weights
            weights = weights / weights.sum()

            for i, (_, row) in enumerate(group.iterrows()):
                channel = row['channel']
                results[channel] = results.get(channel, 0) + weights[i]

    return pd.DataFrame(list(results.items()), columns=['channel', 'credit'])

In [ ]:
# Merge all results
import plotly.express as px


first_touch = first_touch_attribution(journeys)
last_touch = last_touch_attribution(journeys)
linear = linear_attribution(journeys)
time_decay = time_decay_attribution(df)

comparison = first_touch.merge(last_touch, on='channel', how='outer', suffixes=('_first', '_last'))
comparison = comparison.merge(linear, on='channel', how='outer')
comparison = comparison.merge(time_decay, on='channel', how='outer')

comparison.columns = ['channel', 'first_touch', 'last_touch', 'linear', 'time_decay']

comparison = comparison.fillna(0)

print(comparison)


# -------- First Touch --------
fig_first = px.bar(
    first_touch.sort_values(by='credit', ascending=False),
    x='channel',
    y='credit',
    title='First-Touch Attribution',
)
fig_first.show()


# -------- Last Touch --------
fig_last = px.bar(
    last_touch.sort_values(by='credit', ascending=False),
    x='channel',
    y='credit',
    title='Last-Touch Attribution',
)
fig_last.show()


# -------- Linear --------
fig_linear = px.bar(
    linear.sort_values(by='credit', ascending=False),
    x='channel',
    y='credit',
    title='Linear Attribution',
)
fig_linear.show()


# -------- Time Decay --------
fig_time = px.bar(
    time_decay.sort_values(by='credit', ascending=False),
    x='channel',
    y='credit',
    title='Time-Decay Attribution',
)
fig_time.show()

          channel  first_touch  last_touch    linear  time_decay
0          Direct            1           2  2.283333    2.331419
1           Email            4           2  2.600000    2.611221
2      Google Ads            4           4  4.666667    4.484035
3  Organic Search            3           4  3.616667    3.864303
4        Referral            7           5  5.650000    5.371274
5    Social Media            1           3  1.183333    1.337749


In [ ]:
import plotly.express as px

comparison_melted = comparison.melt(id_vars='channel', var_name='model', value_name='credit')

fig = px.bar(
    comparison_melted,
    x='channel',
    y='credit',
    color='model',
    barmode='group',
    title="Attribution Model Comparison"
)

fig.show()

In [ ]:
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 74.0 MB/s eta 0:00:00


In [ ]:
!pip install pyngrok

from dash import Dash, dcc, html
from pyngrok import ngrok

ngrok.set_auth_token("3Dt6Ut5yGVyuNNLSyEuqIzvBKmb_7FT3ZdEP84AFWrPn9Lfjf")

# Create figure
fig = px.bar(
    comparison_melted,
    x='channel',
    y='credit',
    color='model',
    barmode='group',
    title="Marketing Attribution Comparison"
)

# Create app (ONLY ONCE)
app = Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1("Marketing Attribution Dashboard"),
    dcc.Graph(figure=fig)
])

# Open ngrok tunnel
public_url = ngrok.connect(8050)
print("🔗 Open this URL:", public_url)

# Run app
app.run(port=8050)

🔗 Open this URL: NgrokTunnel: "https://dilute-snoring-transpose.ngrok-free.dev" -> "http://localhost:8050"
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/May/2026 13:11:42] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 13:11:43] "GET /_dash-component-suites/dash/html/dash_html_components.v4_1_0m1779109624.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 13:11:43] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1779109624.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 13:11:43] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_1_0m1779109624.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 13:11:43] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1779109624.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 13:11:43] "GET /_dash-component-suites/dash/dash_table/bundle.v7_1_0m17791